# Avaliador de Vendas no Colab — 1 clique (Run all)

Para quem nunca usou Colab:
1. `Tempo de execução → Executar tudo`.
2. Quando pedir, cole sua `GROQ_API_KEY` (console.groq.com, grátis). Pode pular as outras.
3. Aguarde a tabela + gráfico. Nada é salvo no GitHub, só na memória desta sessão.
4. Ao terminar em PC compartilhado: `Tempo de execução → Desconectar`.

Ordem LLM: **Groq → Ollama Cloud → OpenRouter** (Groq é o mais rápido).

In [ ]:
# 0) Colab via GitHub: se abriu só o notebook (sem src/), clona o repo
from pathlib import Path
import os
if not Path('src').exists() and not Path('avaliador-vendas').exists():
    print('Clonando repo...')
    !git clone https://github.com/fcervan/avaliador-vendas.git
if Path('avaliador-vendas').exists() and not Path('src').exists():
    %cd avaliador-vendas
print('cwd:', Path.cwd())
print('src existe:', Path('src').exists())


In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd()
if (ROOT / '../requirements.txt').exists(): ROOT = (ROOT / '..').resolve()
if (ROOT / 'avaliador-vendas/requirements.txt').exists(): ROOT = ROOT / 'avaliador-vendas'
print('ROOT:', ROOT)
!pip install -q -r "$ROOT/requirements.txt"
sys.path.insert(0, str(ROOT / 'src'))

In [ ]:
import os
EM_COLAB = False
try:
    from google.colab import userdata; EM_COLAB = True
except ImportError:
    pass
def pegar_chave(nome):
    if EM_COLAB:
        try:
            v = userdata.get(nome)
            if v: print(f'{nome}: via Secrets ✔'); return v
        except Exception: pass
    if os.getenv(nome): print(f'{nome}: via ambiente ✔'); return os.getenv(nome)
    from getpass import getpass
    return getpass(f'Cole {nome} (oculto, Enter p/ pular): ').strip()
for k in ['GROQ_API_KEY','OLLAMA_CLOUD_API_KEY','OPENROUTER_API_KEY']:
    v = pegar_chave(k)
    if v: os.environ[k] = v
print('OK — precisa de pelo menos 1 chave.')

In [ ]:
from llm_client import get_llm
from graph import grade_transcricao
import pandas as pd, matplotlib.pyplot as plt
llm = get_llm(); print('LLM:', type(llm).__name__)
df = pd.read_csv(ROOT / 'data/exemplos.csv')
resultados = []
for _, row in df.iterrows():
    r = grade_transcricao(row['transcricao'], llm)
    resultados.append({'id': row['id'], 'final': r['final_score'], 'veredito': r['veredito'],
        'saudacao': r['saudacao_score']*10, 'descoberta': r['descoberta_score']*10,
        'apresentacao': r['apresentacao_score']*10, 'fechamento': r['fechamento_score']*10})
res = pd.DataFrame(resultados)
print(res[['id','final','veredito']].to_string(index=False))
res.plot(x='id', y=['saudacao','descoberta','apresentacao','fechamento','final'], kind='bar')
plt.title('Avaliador de Vendas por critério (0-10)'); plt.tight_layout(); plt.show()